# Customer Retention Prioritizer

**Decision question:** If the retailer has limited retention budget, which customers should it contact first?

This notebook uses the public **UCI Online Retail** dataset and demonstrates an end-to-end workflow:
**data ingestion → quality checks → SQL/KPIs → customer analytics → segmentation → retention prioritization → management recommendations**.


In [ ]:
# Run once if needed:
# %pip install -r requirements.txt

import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.retention_analysis import (
    load_raw_data,
    clean_transactions,
    headline_kpis,
    monthly_kpis,
    build_customer_rfm,
    build_segment_summary,
    build_cohort_retention,
    save_sqlite,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 1. Load the real public dataset

The loader automatically downloads the official UCI archive the first time you run it and caches the Excel file under `data/`.


In [ ]:
raw = load_raw_data()
print(f"Raw rows: {len(raw):,}")
raw.head()


## 2. Data-quality audit and cleaning

Rather than silently dropping records, we first quantify data-quality issues. Completed customer-level sales are then retained for behavioral analysis.


In [ ]:
sales, audit = clean_transactions(raw)
audit


In [ ]:
print(f"Usable completed sales rows: {len(sales):,}")
print(f"Date range: {sales['InvoiceDate'].min():%Y-%m-%d} to {sales['InvoiceDate'].max():%Y-%m-%d}")
sales.head()


## 3. Management KPIs

These are the first numbers a manager would want before looking at customer-level detail.


In [ ]:
kpis = headline_kpis(sales)
display(kpis.to_frame("Value"))


In [ ]:
monthly = monthly_kpis(sales)
monthly.tail()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly["InvoiceMonth"], monthly["Revenue"], marker="o")
ax.set_title("Monthly Revenue")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue (£)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


## 4. Demonstrate SQL, not only pandas

We store the cleaned transaction table in SQLite and produce a monthly management report with SQL.


In [ ]:
db_path = save_sqlite(sales)
query = '''
SELECT
    substr(InvoiceDate, 1, 7) AS month,
    ROUND(SUM(Revenue), 2) AS revenue_gbp,
    COUNT(DISTINCT InvoiceNo) AS orders,
    COUNT(DISTINCT CustomerID) AS customers,
    ROUND(SUM(Revenue) / COUNT(DISTINCT InvoiceNo), 2) AS aov_gbp
FROM transactions
GROUP BY substr(InvoiceDate, 1, 7)
ORDER BY month;
'''

with sqlite3.connect(db_path) as conn:
    sql_monthly = pd.read_sql_query(query, conn)

sql_monthly.tail()


## 5. Build customer-level RFM features

For every customer we calculate:

- **Recency:** days since last purchase
- **Frequency:** number of unique orders
- **Monetary:** historical revenue
- **AOV:** average order value
- **Retention Priority:** an interpretable ranking combining value, frequency and lapse


In [ ]:
customer = build_customer_rfm(sales)
customer.head(10)


## 6. Who creates the most value?

A useful segmentation should change a decision, not just assign labels.


In [ ]:
segment_summary = build_segment_summary(customer)
segment_summary


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
plot_df = segment_summary.sort_values("Revenue")
ax.barh(plot_df["Segment"], plot_df["Revenue"])
ax.set_title("Historical Revenue by Customer Segment")
ax.set_xlabel("Revenue (£)")
plt.tight_layout()
plt.show()


## 7. Retention target shortlist

The project intentionally separates **customer value** from **customer inactivity**.

A `PriorityTarget` is:
- top quartile in historical value; and
- at least median in inactivity.

The ranking then combines:

- 50% historical value
- 20% purchase frequency
- 30% inactivity/lapse

This is deliberately transparent. It is a prioritization rule—not a claim that these customers are guaranteed to churn.


In [ ]:
targets = (
    customer.loc[customer["PriorityTarget"]]
    .sort_values("RetentionPriority", ascending=False)
    [[
        "CustomerID", "Country", "Recency", "Frequency",
        "Monetary", "AverageOrderValue", "Segment", "RetentionPriority"
    ]]
)

print(f"Priority customers: {len(targets):,}")
targets.head(20)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sample = customer.nlargest(min(1500, len(customer)), "Monetary")
ax.scatter(sample["Recency"], sample["Monetary"], alpha=0.35)
ax.set_title("Customer Value vs. Inactivity")
ax.set_xlabel("Days since last purchase")
ax.set_ylabel("Historical customer value (£)")
plt.tight_layout()
plt.show()


## 8. Cohort retention

This adds a more advanced behavioral view: are newer customer cohorts continuing to purchase in later months?


In [ ]:
retention = build_cohort_retention(sales)
retention.iloc[:8, :8]


## 9. Management output

A strong portfolio project should end with a recommendation rather than an EDA conclusion.

After running the data, summarize:

1. **Current performance:** revenue, orders, customers, AOV, repeat rate.
2. **Value concentration:** which customer segments contribute most revenue?
3. **Retention opportunity:** how many high-value customers are becoming inactive?
4. **Priority list:** which customers should receive retention budget first?
5. **Action:** what intervention would you test on the priority segment?
6. **Measurement:** compare reactivation / incremental revenue against a control group.

### Important analytical limitation

The transaction history tells us **who is valuable and inactive**, but does not prove why they stopped buying. The retention priority score should therefore be used to design a test, not presented as causal truth.


## 10. Turn it into a small product

After the notebook works, run:

```bash
streamlit run app.py
```

The app lets a user filter by country and customer segment, inspect KPIs, and see the recommended retention target list.
